In [ ]:
!pip install torch torchvision torchaudio transformers datasets scikit-learn streamlit flask

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 101.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 89.8 MB/s eta 0:00:00


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification
from transformers import Trainer, TrainingArguments, pipeline

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving IMDB Dataset.csv to IMDB Dataset.csv


In [ ]:
df = pd.read_csv("IMDB Dataset.csv")  
df['sentiment'] = df['sentiment'].map({"positive": 1, "negative": 0})  

train_texts, test_texts, train_labels, test_labels = train_test_split(
    df["review"], df["sentiment"], test_size=0.2, random_state=42
)

In [ ]:
!git clone https://huggingface.co/distilbert-base-uncased

Cloning into 'distilbert-base-uncased'...
remote: Enumerating objects: 67, done.
remote: Counting objects: 100% (3/3), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 67 (delta 0), reused 0 (delta 0), pack-reused 64 (from 1)
Unpacking objects: 100% (67/67), 310.53 KiB | 639.00 KiB/s, done.
Filtering content: 100% (5/5), 1.42 GiB | 43.66 MiB/s, done.


In [ ]:
tokenizer = DistilBertTokenizerFast.from_pretrained("/content/distilbert-base-uncased")
model = DistilBertForSequenceClassification.from_pretrained("/content/distilbert-base-uncased")

train_encodings = tokenizer(list(train_texts), truncation=True, padding=True)
test_encodings = tokenizer(list(test_texts), truncation=True, padding=True)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at /content/distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
import torch

class IMDbDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels.iloc[idx])
        return item

train_dataset = IMDbDataset(train_encodings, train_labels)
test_dataset = IMDbDataset(test_encodings, test_labels)

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    # evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    report_to=[]
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = logits.argmax(axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average="binary")
    acc = accuracy_score(labels, predictions)
    return {"accuracy": acc, "precision": precision, "recall": recall, "f1": f1}

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

In [ ]:
import os
os.environ["WANDB_DISABLED"] = "true"

In [ ]:
trainer.train()

Step,Training Loss
500,0.152200
1000,0.263200
1500,0.228200
2000,0.212600
2500,0.207800
3000,0.132600
3500,0.132700
4000,0.127700
4500,0.123500
5000,0.130400


TrainOutput(global_step=5000, training_loss=0.17109999694824218, metrics={'train_runtime': 3602.3107, 'train_samples_per_second': 22.208, 'train_steps_per_second': 1.388, 'total_flos': 1.059739189248e+16, 'train_loss': 0.17109999694824218, 'epoch': 2.0})

In [ ]:
trainer.save_model("./sentiment_model")
tokenizer.save_pretrained("./sentiment_model")

classifier = pipeline("sentiment-analysis", model="./sentiment_model", tokenizer="./sentiment_model")

Device set to use cuda:0



Enter a movie review (or type 'exit' to quit): It was okay
Sentiment: LABEL_0 (confidence: 0.5768)

Enter a movie review (or type 'exit' to quit): exit


In [ ]:
!zip -r sentiment_model.zip sentiment_model

  adding: sentiment_model/ (stored 0%)
  adding: sentiment_model/tokenizer.json (deflated 71%)
  adding: sentiment_model/tokenizer_config.json (deflated 75%)
  adding: sentiment_model/training_args.bin (deflated 54%)
  adding: sentiment_model/model.safetensors (deflated 8%)
  adding: sentiment_model/special_tokens_map.json (deflated 42%)
  adding: sentiment_model/config.json (deflated 45%)
  adding: sentiment_model/vocab.txt (deflated 53%)


In [ ]:
from google.colab import files
files.download("sentiment_model.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
label2id = {"NEGATIVE": 0, "POSITIVE": 1}

while True:
  review = input("\nEnter a movie review (or type 'exit' to quit): ")
  if review.lower() == "exit":
    break
  result = classifier(review)[0]
  if(result['label'] == "LABEL_0"):
    result['label'] = "NEGATIVE"
  else:
    result['label'] = "POSITIVE"
  print(f"Sentiment: {result['label']} (confidence: {result['score']:.4f})")


Enter a movie review (or type 'exit' to quit): It was excellent
Sentiment: POSITIVE (confidence: 0.9857)

Enter a movie review (or type 'exit' to quit): It was good
Sentiment: POSITIVE (confidence: 0.9680)

Enter a movie review (or type 'exit' to quit): It was nice
Sentiment: POSITIVE (confidence: 0.8513)

Enter a movie review (or type 'exit' to quit): It was good enough
Sentiment: POSITIVE (confidence: 0.7947)

Enter a movie review (or type 'exit' to quit): It was okay
Sentiment: NEGATIVE (confidence: 0.5768)

Enter a movie review (or type 'exit' to quit): It was not bad
Sentiment: POSITIVE (confidence: 0.7025)

Enter a movie review (or type 'exit' to quit): It was fine
Sentiment: POSITIVE (confidence: 0.8236)

Enter a movie review (or type 'exit' to quit): It was bad


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Sentiment: NEGATIVE (confidence: 0.9291)

Enter a movie review (or type 'exit' to quit): It was not quite bad
Sentiment: NEGATIVE (confidence: 0.7089)

Enter a movie review (or type 'exit' to quit): It was not bad
Sentiment: POSITIVE (confidence: 0.7025)

Enter a movie review (or type 'exit' to quit): exit
